# USD/CAD — Rolling-Origin Starter-Agent Backtest

This notebook evaluates three information settings for the USD/CAD forecasting agent across rolling forecast origins and three horizons: **1, 5, and 21 business days**.

**Target:** `usdcad_logret_{h}b = log(DEXCAUS[t+h] / DEXCAUS[t])`.

Because FRED `DEXCAUS` is Canadian dollars per U.S. dollar, a positive target return means USD/CAD rises (USD strengthens relative to CAD); a negative return means CAD strengthens relative to USD.

### Experimental conditions
- **baseline:** target history only
- **fred_covariates:** target history + the existing leak-safe FRED covariate panel
- **agentic_search:** target history + FRED covariates + cutoff-aware external research

The backtest keeps the forecast origin, target, horizon, and evaluation procedure aligned across conditions. For each origin, the model only receives the `ForecastContext` available at that `as_of` date.

In [2]:
import warnings
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import sys
warnings.filterwarnings("ignore")

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return here


ROOT = _repo_root()
IMPLEMENTATIONS = ROOT / "implementations"
if str(IMPLEMENTATIONS) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATIONS))
load_dotenv(ROOT / ".env", override=False)  # LLMP rows call the Vector proxy — need PROXY_* set

AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"
RUN_AGENT = True

from usdcad_forecasting import DEFAULT_COVARIATE_SERIES_IDS
from usdcad_forecasting.starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)

print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)
print("Default covariates:", DEFAULT_COVARIATE_SERIES_IDS)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview
Default covariates: ['usdcad_log_ret_1b_l1b', 'fed_funds_level_l1b', 'ust2y_level_l1b', 'ust10y_level_l1b', 'ust2y10y_spread_l1b', 'cpi_mom_logdiff_l1b', 'unemployment_rate_l1b', 'oil_log_ret_1b_l1b']


---
## 1. Meet your agent

Search is enabled by default; code execution is optional. The forecasting skill is always loaded.

---
## 3. Rolling-origin backtest

Each forecast origin is treated as a historical information boundary. The agent receives `svc.context(as_of=origin)` and is scored against the realized target after the requested horizon.

Set `N_ORIGINS` to control runtime. A value of 20 gives 20 origins per horizon. The search condition is the most expensive because it performs external research at each origin.

**Important:** keep external research cutoff-aware. The research skill instructs the agent not to use information published after the forecast origin; inspect search/retrieval behavior before treating this as a fully leakage-free historical backtest.

In [1]:
from datetime import datetime, timezone
import numpy as np
from aieng.forecasting.evaluation.task import ForecastingTask
from usdcad_forecasting import build_usdcad_multivariate_service, usdcad_logret_series_id

HORIZONS = [1, 5, 21]
N_ORIGINS = 20
RUN_BACKTEST = True

EXPERIMENTS = {
    "baseline": {
        "description": "Target history only",
        "enable_search": False,
        "covariates": [],
    },
    "fred_covariates": {
        "description": "Target history + FRED covariates",
        "enable_search": False,
        "covariates": DEFAULT_COVARIATE_SERIES_IDS,
    },
    "agentic_search": {
        "description": "Target history + FRED covariates + cutoff-aware search",
        "enable_search": True,
        "covariates": DEFAULT_COVARIATE_SERIES_IDS,
    },
}

agents = {}
for name, settings in EXPERIMENTS.items():
    cfg = build_starter_agent_config(
        model=AGENT_MODEL,
        enable_search=settings["enable_search"],
        enable_code_exec=False,
    )
    agents[name] = build_starter_agent_predictor(
        cfg,
        covariate_series_ids=settings["covariates"],
    )

print(f"Horizons: {HORIZONS}")
print(f"Origins per horizon: {N_ORIGINS}")
print(f"Conditions: {list(agents)}")

ModuleNotFoundError: No module named 'usdcad_forecasting'

In [ ]:
def _resolved_origins(series, horizon, n_origins):
    ts = pd.to_datetime(series["timestamp"]).drop_duplicates().sort_values()
    return ts.iloc[:-horizon].tolist()[-n_origins:]

def _actual_for_origin(full, origin, horizon):
    future_ts = pd.Timestamp(origin) + pd.offsets.BDay(horizon)
    rows = full[full["timestamp"] >= future_ts]
    return None if rows.empty else float(rows["value"].iloc[0])

def _score_prediction(pred, actual):
    fc = pred.payload
    point = float(fc.point_forecast)
    lo = float(fc.quantiles[0.10])
    hi = float(fc.quantiles[0.90])
    return {
        "forecast": point,
        "actual": actual,
        "error": point - actual,
        "absolute_error": abs(point - actual),
        "squared_error": (point - actual) ** 2,
        "direction_correct": (point >= 0) == (actual >= 0),
        "coverage_80": lo <= actual <= hi,
        "interval_width_80": hi - lo,
        "lower_80": lo,
        "upper_80": hi,
    }

if RUN_BACKTEST:
    svc = build_usdcad_multivariate_service(covariate_series_ids=DEFAULT_COVARIATE_SERIES_IDS)
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    backtest_rows = []

    for horizon in HORIZONS:
        target_id = usdcad_logret_series_id(horizon)
        full = svc.get_series(target_id, as_of=now)
        full["timestamp"] = pd.to_datetime(full["timestamp"])
        full = full.sort_values("timestamp").reset_index(drop=True)
        origins = _resolved_origins(full, horizon, N_ORIGINS)
        print(f"\n===== HORIZON {horizon} BUSINESS DAYS | {len(origins)} ORIGINS =====")

        task = ForecastingTask(
            task_id=f"usdcad_logret_{horizon}b",
            target_series_id=target_id,
            horizons=[horizon],
            frequency="B",
            description=f"USD/CAD cumulative log return, {horizon} business days ahead.",
        )

        for origin_idx, origin in enumerate(origins, start=1):
            actual = _actual_for_origin(full, origin, horizon)
            if actual is None:
                continue
            ctx = svc.context(as_of=pd.Timestamp(origin).to_pydatetime())
            for name, predictor in agents.items():
                print(f"  [{origin_idx}/{len(origins)}] {name} | origin={pd.Timestamp(origin).date()}")
                try:
                    pred = predictor.predict(task, ctx)[0]
                    scored = _score_prediction(pred, actual)
                    scored.update({"horizon": horizon, "origin": pd.Timestamp(origin), "agent": name})
                    backtest_rows.append(scored)
                except Exception as exc:
                    print(f"    ERROR: {type(exc).__name__}: {exc}")

    backtest = pd.DataFrame(backtest_rows)
    if backtest.empty:
        raise RuntimeError("No backtest results were produced.")
    backtest = backtest.sort_values(["horizon", "origin", "agent"]).reset_index(drop=True)
    display(backtest[["horizon","origin","agent","forecast","actual","absolute_error","direction_correct","coverage_80","interval_width_80"]])
else:
    print("RUN_BACKTEST is False — set it to True to run the rolling-origin evaluation.")

---
## 4. Aggregate metrics

MAE and RMSE evaluate point accuracy. Directional accuracy measures whether the forecast sign matches the realized return. 80% coverage measures how often the realized return falls inside the 10th–90th percentile interval; interval width measures sharpness.

In [ ]:
if RUN_BACKTEST:
    summary = (
        backtest.groupby(["horizon", "agent"])
        .agg(
            n=("absolute_error", "size"),
            mae=("absolute_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            mean_error=("error", "mean"),
            directional_accuracy=("direction_correct", "mean"),
            coverage_80=("coverage_80", "mean"),
            mean_interval_width_80=("interval_width_80", "mean"),
        )
        .reset_index()
    )
    summary["mae_pct"] = summary["mae"] * 100
    summary["rmse_pct"] = summary["rmse"] * 100
    summary["mean_error_pct"] = summary["mean_error"] * 100
    summary["directional_accuracy_pct"] = summary["directional_accuracy"] * 100
    summary["coverage_80_pct"] = summary["coverage_80"] * 100
    summary["mean_interval_width_80_pct"] = summary["mean_interval_width_80"] * 100
    display(summary[["horizon","agent","n","mae_pct","rmse_pct","directional_accuracy_pct","coverage_80_pct","mean_interval_width_80_pct","mean_error_pct"]].round(4))
else:
    print("Run the backtest first.")

---
## 5. Interpretation and leakage checks

Do not draw conclusions from a single forecast origin. Compare errors across all rolling origins and all three horizons.

The FRED covariates use the existing USDCAD data module's lagging/anti-leakage rules. For `agentic_search`, verify that retrieved sources were available by the corresponding forecast origin before treating the historical search condition as leakage-free.